# NBA Energy Expenditure — 2015-16 Season
*SportVU tracking data · 25 fps · All available games*

Using raw player position data to answer: **who is going all out, and who is conserving energy?**

Average speed — how fast a player moves when they're on the court — is the most direct proxy for energy output available in this data. A player running at 7 ft/s is burning significantly more energy than one moving at 5.5 ft/s.

**Sections**
1. Max effort vs energy conservation — the full picture
2. Speed leaderboard — who moves fastest, who coasts
3. Total miles — the iron-man stat
4. Fatigue — who slows down the most from Q1 to Q4?
5. Single-game records

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

OUTPUT_CSV = '/Users/willhanley/Desktop/spacing/season_stats.csv'
FPS   = 25
MIN_GAMES = 20

season = pd.read_csv(OUTPUT_CSV)
season = season[season['frames'] >= 300].copy()
season['minutes'] = season['frames'] / FPS / 60

# Per-quarter minutes (requires updated data_collection.ipynb output)
if 'frames_q1' in season.columns:
    season['minutes_q1'] = season['frames_q1'] / FPS / 60
    season['minutes_q4'] = season['frames_q4'] / FPS / 60
else:
    season['minutes_q1'] = np.nan
    season['minutes_q4'] = np.nan

# Season aggregates
agg = (
    season.groupby(['player_id','player_name','team_abbr'])
    .agg(
        avg_speed      = ('avg_speed',    'mean'),
        avg_minutes    = ('minutes',      'mean'),
        avg_minutes_q1 = ('minutes_q1',   'mean'),
        avg_minutes_q4 = ('minutes_q4',   'mean'),
        total_miles    = ('dist_miles',   'sum'),
        avg_q1         = ('q1_avg_speed', 'mean'),
        avg_q4         = ('q4_avg_speed', 'mean'),
        games          = ('game_id',      'nunique'),
    )
    .reset_index()
    .query(f'games >= {MIN_GAMES}')
    .reset_index(drop=True)
)
agg['fatigue_pct'] = (agg['avg_q1'] - agg['avg_q4']) / agg['avg_q1'] * 100

LEAGUE_AVG_SPEED = agg['avg_speed'].mean()
LEAGUE_MED_SPEED = agg['avg_speed'].median()

print(f"Players (>= {MIN_GAMES} games): {len(agg)}")
print(f"League avg speed: {LEAGUE_AVG_SPEED:.2f} ft/s")
print(f"Minutes range:    {agg['avg_minutes'].min():.0f} – {agg['avg_minutes'].max():.0f} min/game")
if 'frames_q1' in season.columns:
    print(f"Q1/Q4 frame data available — fatigue section will use precise quarter-minute filters")

## 1. Max Effort vs Energy Conservation

Plotting every player by **minutes per game** (x) and **avg speed** (y) reveals four types:

| Quadrant | What it means |
|---|---|
| High minutes + High speed | **Workhorses** — going all out, carrying a big load |
| High minutes + Low speed  | **Stars conserving** — elite players deliberately pacing themselves |
| Low minutes + High speed  | **Bench sparks** — explosive in short bursts |
| Low minutes + Low speed   | **Limited role** — not a factor in either direction |

In [ ]:
fig, ax = plt.subplots(figsize=(13, 9))

min_med = agg['avg_minutes'].median()
spd_med = agg['avg_speed'].median()

# Quadrant shading
ax.axvspan(min_med, agg['avg_minutes'].max()+1, ymin=0.5, ymax=1.0,
           color='#d4edda', alpha=0.35, zorder=0)   # high min, high spd — workhorses
ax.axvspan(min_med, agg['avg_minutes'].max()+1, ymin=0.0, ymax=0.5,
           color='#fff3cd', alpha=0.35, zorder=0)   # high min, low spd — conservers
ax.axvspan(agg['avg_minutes'].min()-1, min_med, ymin=0.5, ymax=1.0,
           color='#cce5ff', alpha=0.25, zorder=0)   # low min, high spd — bench sparks
ax.axvspan(agg['avg_minutes'].min()-1, min_med, ymin=0.0, ymax=0.5,
           color='#f8f9fa', alpha=0.35, zorder=0)   # low min, low spd

ax.axvline(min_med, color='#aaa', lw=1, ls='--', zorder=1)
ax.axhline(spd_med, color='#aaa', lw=1, ls='--', zorder=1)

sc = ax.scatter(
    agg['avg_minutes'], agg['avg_speed'],
    c=agg['avg_speed'], cmap='RdYlGn',
    s=agg['games']*0.8, alpha=0.75,
    edgecolors='white', linewidths=0.5, zorder=2
)
plt.colorbar(sc, ax=ax, label='Avg Speed (ft/s)', shrink=0.7)

# Label workhorses (top-right): high min + high speed
workhorses = agg[(agg['avg_minutes'] > min_med) & (agg['avg_speed'] > spd_med)].nlargest(8,'avg_speed')
for _, r in workhorses.iterrows():
    ax.annotate(r['player_name'].split()[-1], (r['avg_minutes'], r['avg_speed']),
                xytext=(5,3), textcoords='offset points', fontsize=7.5,
                color='#1a6e3c', fontweight='bold')

# Label conservers (bottom-right): high min + low speed
conservers = agg[(agg['avg_minutes'] > min_med) & (agg['avg_speed'] < spd_med)].nsmallest(8,'avg_speed')
for _, r in conservers.iterrows():
    ax.annotate(r['player_name'].split()[-1], (r['avg_minutes'], r['avg_speed']),
                xytext=(5,-8), textcoords='offset points', fontsize=7.5,
                color='#856404', fontweight='bold')

# Quadrant labels
ax.text(agg['avg_minutes'].max()-1, agg['avg_speed'].max()-0.02,
        'WORKHORSES', ha='right', fontsize=10, color='#1a6e3c',
        fontweight='bold', alpha=0.7)
ax.text(agg['avg_minutes'].max()-1, agg['avg_speed'].min()+0.02,
        'STARS CONSERVING', ha='right', fontsize=10, color='#856404',
        fontweight='bold', alpha=0.7)
ax.text(agg['avg_minutes'].min()+0.2, agg['avg_speed'].max()-0.02,
        'BENCH SPARKS', fontsize=10, color='#004085',
        fontweight='bold', alpha=0.7)

ax.set_xlabel('Avg Minutes per Game', fontsize=12)
ax.set_ylabel('Avg Speed — ft/s (energy intensity)', fontsize=12)
ax.set_title('Max Effort vs Energy Conservation — 2015-16 Season\n'
             'Each dot = one player · size = games played · dashed lines = league medians',
             fontsize=13, fontweight='bold')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

print("Most energy expended (high minutes + high speed):")
for _, r in workhorses.iterrows():
    print(f"  {r['player_name']:<28} {r['avg_speed']:.2f} ft/s  {r['avg_minutes']:.0f} min/g")
print("\nMost energy conserved (high minutes + low speed):")
for _, r in conservers.iterrows():
    print(f"  {r['player_name']:<28} {r['avg_speed']:.2f} ft/s  {r['avg_minutes']:.0f} min/g")

## 2. Speed Leaderboard

Deviation from league median. Players to the right are moving faster than the average player; to the left, slower. Filtered to players with 20+ games.

In [ ]:
# Show top 20 and bottom 20 by avg_speed, as deviation from league median
top20    = agg.nlargest(20, 'avg_speed')
bot20    = agg.nsmallest(20, 'avg_speed')
lollipop = pd.concat([bot20, top20]).drop_duplicates('player_id').sort_values('avg_speed').reset_index(drop=True)
lollipop['dev'] = lollipop['avg_speed'] - LEAGUE_MED_SPEED

fig, ax = plt.subplots(figsize=(11, 11))

colors = ['#ce1141' if d < 0 else '#1d428a' for d in lollipop['dev']]
y = np.arange(len(lollipop))

# Stems from zero
ax.hlines(y, 0, lollipop['dev'], colors=colors, lw=2, alpha=0.7)
# Dots
ax.scatter(lollipop['dev'], y, color=colors, s=55, zorder=3)
# Value labels
for i, row in lollipop.iterrows():
    offset = 0.008 if row['dev'] >= 0 else -0.008
    ha     = 'left'  if row['dev'] >= 0 else 'right'
    ax.text(row['dev'] + offset, i, f"{row['dev']:+.2f}",
            va='center', ha=ha, fontsize=7.5, color='#333')

ax.axvline(0, color='black', lw=1.2)
ax.set_yticks(y)
ax.set_yticklabels(
    [f"{r['player_name']} ({r['team_abbr']})".ljust(30) for _, r in lollipop.iterrows()],
    fontsize=8
)
ax.set_xlabel('Deviation from league median speed (ft/s)', fontsize=11)
ax.set_title('Speed Leaderboard — Top 20 & Bottom 20 vs League Median\n'
             'Blue = above median (more energy) · Red = below (less energy)',
             fontsize=12, fontweight='bold')

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#1d428a', label='Above median — more energy'),
    Patch(color='#ce1141', label='Below median — less energy'),
], fontsize=9, loc='lower right')

ax.spines[['top','right','left']].set_visible(False)
ax.tick_params(axis='y', length=0)
ax.set_xlim(lollipop['dev'].min() - 0.15, lollipop['dev'].max() + 0.15)
plt.tight_layout()
plt.show()

## 3. Total Miles — The Iron-Man Stat

Total distance run across the whole season. Unlike avg speed, this rewards players who play heavy minutes AND move hard — the true workload leaders.

In [ ]:
top_miles = agg.nlargest(25, 'total_miles').sort_values('total_miles', ascending=True)

fig, ax = plt.subplots(figsize=(10, 9))

norm  = plt.Normalize(top_miles['avg_speed'].min(), top_miles['avg_speed'].max())
cmap  = plt.cm.RdYlGn
colors = [cmap(norm(v)) for v in top_miles['avg_speed']]

bars = ax.barh(top_miles['player_name'], top_miles['total_miles'],
               color=colors, edgecolor='white', height=0.7)

for bar, row in zip(bars, top_miles.itertuples()):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f"{row.total_miles:.0f} mi  ({row.avg_speed:.2f} ft/s)",
            va='center', fontsize=8, color='#333')

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, label='Avg Speed (ft/s) — green = high energy, red = conserving', shrink=0.6)

ax.set_xlabel('Total Miles, Full Season', fontsize=11)
ax.set_title('Total Miles Run — Top 25 Players, 2015-16 Season\n'
             'Color = avg speed: green players are covering ground at high intensity',
             fontsize=12, fontweight='bold')
ax.spines[['top','right']].set_visible(False)
ax.set_xlim(0, top_miles['total_miles'].max() * 1.18)
plt.tight_layout()
plt.show()

## 4. Fatigue — Who Slows Down from Q1 to Q4?

Each dot is a player's **Q1 average speed vs Q4 average speed** across the season. The diagonal line means no change. Points below the line slowed down in the 4th quarter; points above sped up.

Filtered to players averaging **20+ minutes per game** — ensures we're looking at players who actually played meaningful time in both quarters, not garbage-time appearances.

In [ ]:
# Build fatigue dataset
# If new per-quarter frame data is available: only include game rows where
# the player logged 5+ min in BOTH Q1 and Q4, so averages aren't skewed by
# garbage-time appearances. Falls back to avg_minutes >= 20 filter otherwise.
if 'frames_q1' in season.columns:
    # Game-level filter: meaningful Q1 and Q4 time
    fat_games = season[
        (season['frames_q1'] >= 125) &   # 5+ min in Q1
        (season['frames_q4'] >= 125)      # 5+ min in Q4
    ].copy()
    fat_agg = (
        fat_games.groupby(['player_id','player_name','team_abbr'])
        .agg(
            avg_q1 = ('q1_avg_speed', 'mean'),
            avg_q4 = ('q4_avg_speed', 'mean'),
            games  = ('game_id',      'nunique'),
        )
        .reset_index()
        .query('games >= 10')   # need enough qualifying games
        .reset_index(drop=True)
    )
    # Attach avg_minutes from the main agg for context
    fat = fat_agg.merge(agg[['player_id','avg_minutes']], on='player_id', how='left')
    filter_note = 'Filtered to games with 5+ min in both Q1 and Q4 (10+ qualifying games)'
else:
    # Fallback: season-level avg_minutes >= 20
    fat = agg[(agg['avg_minutes'] >= 20)].dropna(subset=['avg_q1','avg_q4']).copy()
    filter_note = 'Filtered to players averaging 20+ min/game (re-run data_collection.ipynb for precise Q1/Q4 filters)'

fat['fatigue_pct'] = (fat['avg_q1'] - fat['avg_q4']) / fat['avg_q1'] * 100
fat = fat.dropna(subset=['avg_q1','avg_q4','fatigue_pct'])

print(f"Players in fatigue chart: {len(fat)}")
print(f"Note: {filter_note}")

fig, ax = plt.subplots(figsize=(10, 10))

norm  = plt.Normalize(fat['fatigue_pct'].min(), fat['fatigue_pct'].max())
cmap  = plt.cm.RdYlGn_r   # red = big drop, green = speeds up
colors = [cmap(norm(v)) for v in fat['fatigue_pct']]

ax.scatter(fat['avg_q1'], fat['avg_q4'],
           c=fat['fatigue_pct'], cmap='RdYlGn_r',
           s=50, alpha=0.75, edgecolors='white', linewidths=0.5,
           norm=norm)

# Diagonal = no change
lims = [min(fat['avg_q1'].min(), fat['avg_q4'].min()) - 0.05,
        max(fat['avg_q1'].max(), fat['avg_q4'].max()) + 0.05]
ax.plot(lims, lims, color='black', lw=1.2, ls='--', alpha=0.5, label='No change')

sm = plt.cm.ScalarMappable(cmap='RdYlGn_r', norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, label='% Speed Drop Q1→Q4  (red = big drop, green = speeds up)')

# Label most fatigued (biggest % drop)
most_tired = fat.nlargest(8, 'fatigue_pct')
for _, r in most_tired.iterrows():
    ax.annotate(r['player_name'].split()[-1],
                (r['avg_q1'], r['avg_q4']),
                xytext=(5, -8), textcoords='offset points',
                fontsize=7.5, color='#a50026', fontweight='bold')

# Label most clutch (negative = sped up)
most_clutch = fat.nsmallest(6, 'fatigue_pct')
for _, r in most_clutch.iterrows():
    ax.annotate(r['player_name'].split()[-1],
                (r['avg_q1'], r['avg_q4']),
                xytext=(5, 5), textcoords='offset points',
                fontsize=7.5, color='#1a6e3c', fontweight='bold')

ax.fill_between(lims, lims, min(lims), alpha=0.04, color='red', label='Slower in Q4')
ax.fill_between(lims, lims, max(lims), alpha=0.04, color='green', label='Faster in Q4')

ax.set_xlabel('Q1 Avg Speed (ft/s)', fontsize=12)
ax.set_ylabel('Q4 Avg Speed (ft/s)', fontsize=12)
ax.set_title('Fatigue: Q1 vs Q4 Avg Speed\n' + filter_note,
             fontsize=12, fontweight='bold')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

print("\nMost fatigued (biggest % speed drop Q1→Q4):")
for _, r in most_tired.iterrows():
    print(f"  {r['player_name']:<28} Q1: {r['avg_q1']:.2f}  Q4: {r['avg_q4']:.2f}  drop: {r['fatigue_pct']:+.1f}%")
print("\nSpeeds up in Q4 (clutch-time mode):")
for _, r in most_clutch.iterrows():
    print(f"  {r['player_name']:<28} Q1: {r['avg_q1']:.2f}  Q4: {r['avg_q4']:.2f}  change: {r['fatigue_pct']:+.1f}%")

## 5. Single-Game Records

In [ ]:
s  = season
sn = season.dropna(subset=['fatigue_pct'])

records = [
    ('Fastest top speed in a game',   s.loc[s['top_speed'].idxmax()],        'top_speed',    '{:.2f} ft/s'),
    ('Most miles in one game',        s.loc[s['dist_miles'].idxmax()],       'dist_miles',   '{:.2f} mi'),
    ('Fewest miles in one game',      s.loc[s['dist_miles'].idxmin()],       'dist_miles',   '{:.2f} mi'),
    ('Highest avg speed in a game',   s.loc[s['avg_speed'].idxmax()],        'avg_speed',    '{:.2f} ft/s'),
    ('Lowest avg speed in a game',    s.loc[s['avg_speed'].idxmin()],        'avg_speed',    '{:.2f} ft/s'),
    ('Biggest Q4 slowdown',           sn.loc[sn['fatigue_pct'].idxmax()],    'fatigue_pct',  '{:+.1f}%'),
    ('Biggest Q4 speed-up',           sn.loc[sn['fatigue_pct'].idxmin()],    'fatigue_pct',  '{:+.1f}%'),
]

print(f"{'Record':<32}  {'Player':<25}  {'Team':>5}  {'Value':>10}")
print('-' * 78)
for label, row, col, fmt in records:
    print(f"{label:<32}  {row['player_name']:<25}  {row['team_abbr']:>5}  {fmt.format(row[col]):>10}")